In [ ]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

from sklearn.tree import DecisionTreeRegressor  #回帰木

In [ ]:
df = pd.read_csv('datafiles/train.csv')

In [ ]:
#明らかに不要な'id'を除く
df = df.drop(['Id'], axis = 1)

In [ ]:
#順序尺度の列に格納されている値を数値に変更
#コードの可読性のために、NAを0に置き換え、より好ましい値ほど大きい値を格納するようにした。

df['LandSlope'] = df['LandSlope'].replace({
    'Gtl': 3,
    'Mod': 2,
    'Sev': 1
})

In [ ]:
#ダミー変数化する行の抜き出し
to_dummy_cols = []
for c in df.columns:
    if type(c) == str:
        to_dummy_cols.append(c)
print(to_dummy_cols)

In [ ]:
'''
#strの特徴量の中にNAが混ざっている列
Alley
MasVnrType
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2
Electrical
FireplaceQu
GarageType
GarageFinish
GarageQual
GarageCond
PoolQC
Fence
MiscFeature

#intの特徴量の中にNAが混ざっている列
LotFrontage  NAを0に変更
MasVnrArea   NAを0に変更
GarageYrBlt  NAを0に変更
'''
#data_description.txt を確認すると、すべての特徴量で'NA'に意味があるようだったので補完
to_NA_cols = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 
    'GarageCond', 'Fence', 'MiscFeature'
]
df[to_NA_cols] = df[to_NA_cols].fillna('NA')
df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']] = df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']].fillna(0.0)


In [ ]:
print(df.columns)

In [ ]:
#float型に変更
not_to_dummy = ['MSSubClass', 'LotFrontage', 
    'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 
    'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
    'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 
    'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice'
]

In [ ]:
df[not_to_dummy] = df[not_to_dummy].astype('float64')

In [ ]:
#ダミー変数化
to_dummy = set(df.columns) - set(not_to_dummy)
to_dummy = list(to_dummy)
for c in to_dummy:
    dummy = pd.get_dummies(df[c], prefix=c, drop_first = True, dtype = int)
    df = pd.concat([df, dummy], axis = 1)
    df = df.drop([c], axis = 1)

In [ ]:
#float型に変更
df = df.astype('float64')

In [ ]:
#説明変数と目的変数のデータフレームを作る
df_y = pd.DataFrame(df['SalePrice'])
df_x = df.drop(['SalePrice'], axis = 1)

#標準化
sc_model=StandardScaler()
sc_model.fit(df_x)
sc_x = sc_model.fit_transform(df_x)

In [ ]:
#回帰木を実践し、結果を比較する。
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
model2.fit(sc_x, df_y)

In [ ]:
#リッジ回帰の係数と切片の確認
coef_df = pd.DataFrame({
    'col': df_x.columns,
    'coef': model2.coef_
})
print(f'係数: {coef_df }')
print(f'切片: {model2.intercept_}')

In [ ]:
coef_df.sort_values('coef', ascending=False)

In [ ]:
'''
NAが複数行にかけて同一の意味をもつもの
・no basementの意味
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2

・no garageの意味
GarageType
GarageFinish
GarageQual
GarageCond
'''
sc_x = pd.DataFrame(sc_x)
sc_x.columns = df_x.columns
for c in sc_x.columns:
    print(c)


In [ ]:
'''
前のセルの結果、下記のセルが重複するNAのダミー変数化であった。

BsmtQual_NA
BsmtCond_NA
BsmtExposure_NA
BsmtFinType1_NA
BsmtFinType2_NA
　→BsmtQual_NAのみ残し、他は削除

GarageCond_NA
GarageType_NA
GarageQual_NA
　→GarageFinish列で同一の意味を表せるので削除
'''
to_drop = ['BsmtCond_NA', 'BsmtExposure_NA', 'BsmtFinType1_NA', 'BsmtFinType2_NA', 'GarageCond_NA', 'GarageType_NA', 'GarageQual_NA']
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

In [ ]:
#回帰木
best_score = 0
best_depth = 0
#深さを１～30まで実験
for i in range(1,31):
    model = DecisionTreeRegressor(max_depth = i, random_state = 0)
    all_result = cross_validate(model, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_score:
        best_score = result
        best_depth = i
print(f'深さ＝{best_depth}　回帰木のスコア＝{best_score}')

model4 = DecisionTreeRegressor(max_depth = best_depth, random_state = 0)
result = cross_validate(model4, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel4のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

In [ ]:
#多重共線性の解消のためにdropメソッドを使う前に、データフレームを保存
df1_all_col = pd.concat([sc_x, df_y], axis = 1)
df1_all_col.to_csv('df1_all_col.csv', index=False)